## 一、总体开发方向

目标是设计一套面向文字冒险/视觉小说的剧情脚本系统，整体参考 Ren’Py，并结合 HTML、Markdown、BBCode 等标记语言的特点。

总体构想可以概括为：

> 使用自定义剧情脚本语言 `.dramex` 编写剧情，经解析或编译后转换为 JSON 等运行时数据，再由 Godot 内部的管理器和播放器执行。

主要特征包括：

- 以顺序剧情为基础
- 支持跳转、分支和脚本文件切换
- 使用主脚本统筹剧情
- 子脚本执行完毕后返回主脚本
- 支持场景、背景、音乐、角色、表情、转场和震动等控制
- 支持对话文本内部的动态效果
- 计划制作 VS Code 插件或其他编辑器插件

## 二、`.dramex` 剧情脚本语言设计

目前已经形成了较明确的脚本语言方向。

### 文件与模块结构

计划使用：

```text
.dramex
```

作为剧情脚本的扩展名，并使用：

```text
__init__.dramex
```

标记一个剧情块或脚本模块。

设想中的特性：

- 一个文件夹可以代表一个剧情块
- 剧情块之间可以嵌套
- `__init__.dramex` 中声明该块需要使用的角色、表情和自定义函数
- 允许通过 `from __init__ import func` 导入函数
- 使用 `#` 添加注释

示意：

```text
character {
    "灵梦": ["idle", "smile", "laugh"],
    "Phoenix": ["idle", "smile", "sad"]
}
```

自定义函数的语法目前倾向于参考 Python：

```text
def func(time, effects):
    ...
```

### 命令分组

计划将命令分为三类：

1. 对话命令

```text
灵梦 我是灵梦
```

2. 效果命令

```text
shake(time=3)
```

3. 句内命令

```text
这是$一段文字${命令 参数}的内容
```

句内命令还会继续分为：

- 包裹式命令：影响一段文字的范围
- 插入式命令：在文本播放到某一位置时执行一次效果

计划支持嵌套包裹式命令，插入式命令则使用方括号：

```text
[shake(time=3)]
```


## 三、计划支持的剧情命令

目前记录过的命令包括：

| 命令 | 用途 |
|---|---|
| `location` / `background` | 更换背景 |
| `music` | 更换背景音乐 |
| `show_character` | 显示或更换角色立绘 |
| `expression` | 切换角色表情或动画 |
| `next_scene` | 切换场景或剧情脚本 |
| `transition` | 播放转场效果 |
| `shake` | 屏幕或镜头震动 |
| `goto` | 跳转到指定 anchor |

命令参数倾向于采用类似 Python 函数的形式：

```text
shake(time=3)
```

对于不需要参数的命令，可以使用列表统一收集。

## 四、文本动画与音效控制

### 语气词音效

这是一个插入式命令：

- 文本播放到指定位置时播放语气词音效
- 播放期间降低打字机音效
- 音效结束后恢复打字机音效
- 音量可以使用默认值，也可以通过参数指定

### 打字机音效控制

这是一个包裹式命令：

- 被包裹的文字播放期间临时调整打字机音效
- 参数可以为正数或负数
- 离开包裹范围后恢复原来的音效参数

## 五、Godot 内部系统架构

多个计划中的管理类：

- `ResourceManager`：管理资源路径、文件类型、文件名和资源类别
- `SceneManager`：预加载、切换场景，并考虑维护场景历史栈
- `ScriptManager`：注册和管理剧情脚本，自动记录 anchor 位置
- `ErrorManager` / `SafeGuard`：检查文件、角色、动画帧等是否存在并报告错误
- `CharacterManager`：管理角色实例、角色表情和立绘变化
- `DialogueManager`：负责对话播放和剧情推进

较完整的架构建议是将对话系统拆分为：

```text
DialogueParser  →  DialoguePlayer  →  各类 Command
```

其中：

- `DialogueParser`：解析 `.dramex` 或 JSON
- `DialogueLine`：保存说话人、文本和命令
- `DialoguePlayer`：管理当前剧情位置和 `next()`
- `Command`：将震动、音效、文本速度等行为交给对应系统执行

还提出了以下架构思想：

- 管理器可以使用 Godot AutoLoad 单例
- 对话数据、角色数据可使用 `Resource` 或 `RefCounted`
- 使用信号解耦逻辑层与 UI 层
- 通过状态模式处理调查、法庭等不同游戏状态
- 未来可能加入 `Evidence` 和 `LogicTree`
- 对插入命令可以考虑 Command Pattern 或 Visitor Pattern
- 尽量通过配置文件初始化，避免在管理器内部硬编码路径

## 六、角色、表情与美术资源规范
### 尺寸规范

- 角色立绘需要保证尺寸一致
- 背景图片也需要保证尺寸一致
- 资源管理器需要统一记录资源路径、类型、名称和类别

### 角色动画

计划中的常用角色动画包括：

- `idle`
- `talking`
- `confident-idle`
- `confident-talking`
- `laugh-talking`

角色表情主要通过帧动画完成：

- 动画可以循环播放
- 可以复用动画
- 可以考虑倒放动画
- 东方角色可能增加符卡效果等特殊动画

## 七、编辑器与开发工具

计划制作剧情脚本插件，功能包括：

- `.dramex` 语法高亮
- 实时渲染剧情脚本
- 标签或 anchor 支持
- 可能集成 VS Code

## 八、当前尚未确定的问题

目前最需要进一步决策的内容有：

1. `.dramex` 是否先编译成 JSON，再由 Godot 运行
2. `.dramex` 的正式语法和词法规则
3. `$...$` 包裹式命令的准确写法
4. 句内命令的嵌套、转义和错误处理
5. 动态变量、条件判断和分支语法
6. `__init__.dramex` 的作用范围及模块导入规则
7. 剧情脚本的自动发现方式，是否需要硬编码路径
8. 主脚本与子脚本之间的调用和返回机制
9. 错误处理是中断游戏、输出日志，还是允许使用备用逻辑
10. `next_scene` 是切换场景、切换脚本，还是同时支持两者